# Alice: symbolic and numerical optimization

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gromicho/teaching/blob/main/foundations/optimization/alice-optimization.ipynb)

**Worked Example** · UvA Business Analytics teaching collection

Work through the questions before running each cell. Explain the result, check its assumptions, and change one input to test your understanding.


## Setup

Colab already provides the general-purpose scientific libraries. This cell ensures the tested Pyomo version; pip keeps an already-installed matching version. It does not replace Colab's NumPy, pandas or Matplotlib just to match the maintenance environment.


In [ ]:
# Use installed packages; install only missing ones, without version pins.
from importlib.util import find_spec
missing_packages = [name for name in ['pyomo'] if find_spec(name) is None]
if missing_packages:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])


## A stable vase
Alice's teaching model describes the centre of gravity as a function of the water height $h$:
$$c(h)=\frac{4\pi h^2+2000}{8\pi h+200},\qquad 0\leq h\leq20.$$
Minimizing this function gives the most stable height within this simplified model. The original photograph is not redistributed because its image licence was not established. The mathematical example does not require it.

Before computing, sketch the trade-off: why could neither an empty nor a full vase be best?


In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
h = sp.Symbol('h', real=True)
c = (4*sp.pi*h**2+2000)/(8*sp.pi*h+200)
derivative = sp.diff(c,h)
stationary = sp.solve(derivative,h)
stationary


In [ ]:
candidates = [0.0,20.0] + [float(v) for v in stationary if v.is_real and 0 <= float(v) <= 20]
objective = sp.lambdify(h,c,'numpy')
best = min(candidates,key=objective)
print('Candidates:', candidates, 'best height:', best)
result = minimize_scalar(objective,bounds=(0,20),method='bounded')
assert result.success
assert abs(result.x-best) < 1e-4


In [ ]:
grid = np.linspace(0,20,201)
plt.plot(grid,objective(grid))
plt.scatter([best],[objective(best)],color='red')
plt.xlabel('Water height h')
plt.ylabel('Centre-of-gravity height c(h)')
plt.show()


## Express the same objective in Pyomo
Pyomo is a modelling language, not a solver. HiGHS is appropriate for the linear and mixed-integer linear examples elsewhere in this collection; it does not solve this rational nonlinear objective. Here we verify Pyomo's expression against the independent one-dimensional SciPy calculation. No job, email address or model is submitted to NEOS or another remote solver.


In [ ]:
import pyomo.environ as pyo
alice = pyo.ConcreteModel('Alice')
alice.h = pyo.Var(bounds=(0,20),initialize=best)
alice.cog = pyo.Objective(expr=(4*np.pi*alice.h**2+2000)/(8*np.pi*alice.h+200))
assert abs(pyo.value(alice.cog)-float(objective(best))) < 1e-9
alice.pprint()


## Why is this a minimum?
Compare every stationary point inside the bounds with the two endpoints. Explain what a derivative test establishes, and what the bounds add. A successful numerical termination alone is not a proof of global optimality for an arbitrary nonlinear problem.
